# CARLA Python API - Project 5: Cooperative Roadside Assistant (V2X / I2V)

This notebook contains exactly 3 sections:
1. **Infrastructure Sensing & World Transformation** (Theory, camera setups, and 2D-to-3D projection layout)
2. **V2X MQTT Server Architecture & Message Serialization** (Live MQTT client setup, network delay emulation, and protocol payloads)
3. **End-to-End System Integration & Occlusion Scenarios** (The runnable project execution with full evaluation metrics)

## CARLA docs
- Main docs: https://carla.readthedocs.io/en/latest/
- Sensors reference: https://carla.readthedocs.io/en/latest/ref_sensors/
- Python API: https://carla.readthedocs.io/en/latest/python_api/

In [ ]:
import carla
import time
import random
import cv2
import queue
import threading
import json
import math
import numpy as np
from datetime import datetime

# Initialize client and connect to the CARLA server daemon
client = carla.Client("localhost", 2000)
client.set_timeout(10.0)
world = client.get_world()
spectator = world.get_spectator()
blueprint_library = world.get_blueprint_library()

In [28]:
import carla
import time
import random
import cv2
import queue
import threading
import math
import numpy as np

# ==============================================================================
# 1. CORE SYSTEM UTILITIES AND INITIALIZATION
# ==============================================================================

def move_spectator_to(transform, spectator, distance=12.0, z=6.0, pitch=-25.0):
    """Utility to orient the editor spectator view behind an active actor."""
    back = transform.location - transform.get_forward_vector() * distance
    loc = carla.Location(back.x, back.y, back.z + z)
    rot = carla.Rotation(pitch=pitch, yaw=transform.rotation.yaw, roll=0.0)
    spectator.set_transform(carla.Transform(loc, rot))

def safe_destroy(actors):
    """Safely removes lists of actors under teardown scenarios."""
    for a in actors:
        if a is not None:
            try:
                a.destroy()
            except RuntimeError:
                pass

# Establish connectivity to the running CARLA simulator server
client = carla.Client("localhost", 2000)
client.set_timeout(20.0)

# ==============================================================================
# 2. LOCAL PERCEPTION SIMULATION LOOP (MONITORING CAMERA FEED)
# ==============================================================================

def local_onboard_camera_loop(cam_sensor, stop_event):
    """Simulates an onboard front-facing safety camera for visual validation."""
    frame_queue = queue.Queue(maxsize=1)
    cam_sensor.listen(lambda img: frame_queue.put(img) if not frame_queue.full() else None)
    
    while not stop_event.is_set():
        try:
            image = frame_queue.get(timeout=0.2)
            arr = np.frombuffer(image.raw_data, dtype=np.uint8)
            arr = np.reshape(arr, (image.height, image.width, 4))
            bgr_frame = arr[:, :, :3].copy()
            
            cv2.putText(bgr_frame, "EGO LOCAL ONBOARD CAMERA - UNASSISTED MODE", 
                        (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
            cv2.imshow("Ego Local View (No V2X)", bgr_frame)
            cv2.waitKey(1)
        except queue.Empty:
            continue
            
    cam_sensor.stop()
    cv2.destroyAllWindows()

# ==============================================================================
# 3. CORE ISOLATED SCENARIO EVALUATION ENGINE
# ==============================================================================

def run_unassisted_scenario(map_name, weather_preset, scenario_id):
    """
    Executes a complete isolated test trial in a specific city/weather configuration
    without any cooperative V2X infrastructure inputs.
    """
    print(f"\n" + "="*70)
    print(f"LAUNCHING SCENARIO {scenario_id}: Map={map_name} | Weather={weather_preset}")
    print("="*70)
    
    # Load requested town configuration
    world = client.load_world(map_name)
    world.set_weather(weather_preset)
    blueprint_library = world.get_blueprint_library()
    spectator = world.get_spectator()
    
    # Enable Synchronous Mode for absolute deterministic execution behavior
    original_settings = world.get_settings()
    settings = world.get_settings()
    settings.synchronous_mode = True
    settings.fixed_delta_seconds = 0.04 # 25 FPS
    world.apply_settings(settings)
    
    trial_actors = []
    stop_signal = threading.Event()
    stop_signal.clear()
    
    # Initialize thread tracker to prevent UnboundLocalError if spawning fails early
    cam_thread = None
    
    # Telemetry indicators
    min_distance_to_target = float("inf")
    collision_detected = False
    speed_at_brake_trigger = 0.0
    brake_timestamp = None
    
    try:
        carla_map = world.get_map()
        spawn_points = carla_map.get_spawn_points()
        
        # 1. SETUP DYNAMIC SPAWNS BASED ON MAP
        if "Town03" in map_name:
            # Explicit intersection arrangement to guarantee an occlusion scenario
            ego_spawn_tf = carla.Transform(carla.Location(x=120.0, y=132.0, z=2.0), carla.Rotation(yaw=0.0))
            vru_spawn_tf = carla.Transform(carla.Location(x=152.0, y=142.0, z=1.5), carla.Rotation(yaw=-90.0))
            vru_direction = carla.Vector3D(0, -1, 0)
        else:
            # Dynamic extraction for alternative maps (like Town05) using native navigation maps
            ego_spawn_tf = spawn_points[0]
            
            # Request a certified walking location from CARLA navigation to guarantee zero collisions
            nav_location = world.get_random_location_from_navigation()
            if nav_location is None:
                # Fallback if navigation mesh isn't loaded completely
                nav_location = spawn_points[min(5, len(spawn_points)-1)].location
                nav_location.z += 1.5
                
            vru_spawn_tf = carla.Transform(nav_location, carla.Rotation(yaw=0.0))
            
            # Point pedestrian toward the vehicle direction dynamically
            vru_direction = (ego_spawn_tf.location - vru_spawn_tf.location).make_unit_vector()

        # 2. SPAWN ACTORS
        tesla_bp = blueprint_library.filter("vehicle.tesla.model3")[0]
        ego_vehicle = world.spawn_actor(tesla_bp, ego_spawn_tf)
        trial_actors.append(ego_vehicle)
        
        ped_bp = blueprint_library.filter("walker.pedestrian.0001")[0]
        vru_actor = world.spawn_actor(ped_bp, vru_spawn_tf)
        trial_actors.append(vru_actor)
        
        # Spawn Onboard Front Verification Camera (Attached to Ego Bumper)
        cam_bp = blueprint_library.find("sensor.camera.rgb")
        cam_bp.set_attribute("image_size_x", "640")
        cam_bp.set_attribute("image_size_y", "480")
        cam_transform = carla.Transform(carla.Location(x=2.0, z=1.3), carla.Rotation(pitch=0.0))
        ego_camera = world.spawn_actor(cam_bp, cam_transform, attach_to=ego_vehicle)
        trial_actors.append(ego_camera)
        
        # 3. RUN PERCEPTION WINDOW THREAD
        cam_thread = threading.Thread(target=local_onboard_camera_loop, args=(ego_camera, stop_signal), daemon=True)
        cam_thread.start()
        
        # Settle physics engine state
        world.tick()
        time.sleep(0.5)
        
        # Direct pedestrian to start walking
        vru_control = carla.WalkerControl(direction=vru_direction, speed=1.5)
        vru_actor.apply_control(vru_control)
        
        # Initial cruise settings
        target_throttle = 0.50
        target_brake = 0.0
        
        start_time = time.time()
        max_duration = 12.0
        
        # 4. EXECUTION SIMULATION LOOP
        while time.time() - start_time < max_duration:
            world.tick()
            
            ego_tf = ego_vehicle.get_transform()
            vru_tf = vru_actor.get_transform()
            move_spectator_to(ego_tf, spectator, distance=15.0, z=6.0, pitch=-22.0)
            
            ego_loc = ego_tf.location
            vru_loc = vru_tf.location
            current_distance = ego_loc.distance(vru_loc)
            
            if current_distance < min_distance_to_target:
                min_distance_to_target = current_distance
                
            if current_distance < 1.95:
                collision_detected = True
                
            # Extract speed telemetry data
            vel = ego_vehicle.get_velocity()
            speed_kmh = 3.6 * math.sqrt(vel.x**2 + vel.y**2 + vel.z**2)
            
            # LOCAL SENSOR PERCEPTION BOUNDS (Line-Of-Sight Emulation Window)
            is_visible_locally = (vru_loc.y - ego_loc.y) < 4.5 and abs(vru_loc.x - ego_loc.x) < 13.0
            
            if is_visible_locally:
                if target_brake == 0.0: # Track exact moment local sensor discovered the danger
                    brake_timestamp = time.time() - start_time
                    speed_at_brake_trigger = speed_kmh
                
                target_throttle = 0.0
                target_brake = 1.0
                world.debug.draw_string(ego_loc + carla.Location(z=2.8), "⚠️ EMERGENCY BRAKE (LOS)", 
                                        life_time=0.05, color=carla.Color(255, 0, 0))
            else:
                world.debug.draw_string(ego_loc + carla.Location(z=2.5), "CRUISE CONTROL ACTIVE", 
                                        life_time=0.05, color=carla.Color(0, 255, 0))
                
            ego_vehicle.apply_control(carla.VehicleControl(throttle=float(target_throttle), brake=float(target_brake), steer=0.0))
            world.debug.draw_string(vru_loc + carla.Location(z=2.0), "WALKER", life_time=0.05, color=carla.Color(0, 255, 255))
            
            time.sleep(0.02)
            
            if target_brake == 1.0 and speed_kmh < 0.1:
                print(">> Ego vehicle reached complete standstill.")
                break
                
    finally:
        # Tear down background operations safely
        stop_signal.set()
        if cam_thread is not None:
            cam_thread.join(timeout=1.5)
        world.apply_settings(original_settings)
        safe_destroy(trial_actors)
        time.sleep(1.0)
        
    return {
        "collision": collision_detected,
        "min_dist": min_distance_to_target,
        "brake_time": brake_timestamp,
        "trigger_speed": speed_at_brake_trigger
    }

# ==============================================================================
# 4. BATCH TRIAL EXECUTION MATRIX AND PROJECT REPORT GENERATOR
# ==============================================================================

# Run Scenario 1: Clear Noon Environment inside Town03
run_1_metrics = run_unassisted_scenario(
    map_name="Town03", 
    weather_preset=carla.WeatherParameters.ClearNoon, 
    scenario_id=1
)

# Run Scenario 2: Hard Rain Sunset environment inside Town05
run_2_metrics = run_unassisted_scenario(
    map_name="Town05", 
    weather_preset=carla.WeatherParameters.HardRainSunset, 
    scenario_id=2
)

# Render Final Quantitative Summary Deliverables Report Table
print("\n" + "="*75)
print("             PROJECT 5 DELIVERABLE: UNASSISTED BASELINE REPORT           ")
print("="*75)
print(f"{'Performance Assessment Metric':<35} | {'Scenario 1 (Town03)':<17} | {'Scenario 2 (Town05)':<15}")
print("-"*75)

outcome_1 = "💥 COLLISION" if run_1_metrics["collision"] else "✅ AVOIDED"
outcome_2 = "💥 COLLISION" if run_2_metrics["collision"] else "✅ AVOIDED"
print(f"{'Safety Validation Outcome':<35} | {outcome_1:<17} | {outcome_2:<15}")

print(f"{'Minimum Spatial Proximity (m)':<35} | {run_1_metrics['min_dist']:<17.2f} | {run_2_metrics['min_dist']:<15.2f}")

time_1 = f"{run_1_metrics['brake_time']:.2f}s" if run_1_metrics['brake_time'] else "N/A"
time_2 = f"{run_2_metrics['brake_time']:.2f}s" if run_2_metrics['brake_time'] else "N/A"
print(f"{'Local Perception Discovery Time':<35} | {time_1:<17} | {time_2:<15}")

speed_1 = f"{run_1_metrics['trigger_speed']:.1f} km/h" if run_1_metrics['brake_time'] else "N/A"
speed_2 = f"{run_2_metrics['trigger_speed']:.1f} km/h" if run_2_metrics['brake_time'] else "N/A"
print(f"{'Speed at Initial Brake Trigger':<35} | {speed_1:<17} | {speed_2:<15}")
print("="*75)
print(">> Project validation script run finished successfully.")


LAUNCHING SCENARIO 1: Map=Town03 | Weather=WeatherParameters(cloudiness=5.000000, precipitation=0.000000, precipitation_deposits=0.000000, wind_intensity=10.000000, sun_azimuth_angle=-1.000000, sun_altitude_angle=45.000000, fog_density=2.000000, fog_distance=0.750000, fog_falloff=0.100000, wetness=0.000000, scattering_intensity=1.000000, mie_scattering_scale=0.030000, rayleigh_scattering_scale=0.033100, dust_storm=0.000000)

LAUNCHING SCENARIO 2: Map=Town05 | Weather=WeatherParameters(cloudiness=100.000000, precipitation=100.000000, precipitation_deposits=90.000000, wind_intensity=100.000000, sun_azimuth_angle=-1.000000, sun_altitude_angle=15.000000, fog_density=7.000000, fog_distance=0.750000, fog_falloff=0.100000, wetness=0.000000, scattering_intensity=1.000000, mie_scattering_scale=0.030000, rayleigh_scattering_scale=0.033100, dust_storm=0.000000)

             PROJECT 5 DELIVERABLE: UNASSISTED BASELINE REPORT           
Performance Assessment Metric       | Scenario 1 (Town03) | S

In [1]:
import carla
import time
import random
import cv2
import queue
import threading
import math
import numpy as np

# ==============================================================================
# 1. CORE UTILITIES AND RSU TRANSFORMS
# ==============================================================================

def move_spectator_to(transform, spectator, distance=12.0, z=6.0, pitch=-25.0):
    """Utility to orient the editor spectator view behind an active actor."""
    back = transform.location - transform.get_forward_vector() * distance
    loc = carla.Location(back.x, back.y, back.z + z)
    rot = carla.Rotation(pitch=pitch, yaw=transform.rotation.yaw, roll=0.0)
    spectator.set_transform(carla.Transform(loc, rot))

def safe_destroy(actors):
    """Safely removes lists of actors under teardown scenarios."""
    for a in actors:
        if a is not None:
            try:
                a.destroy()
            except RuntimeError:
                pass

# Connect to CARLA Simulator
client = carla.Client("localhost", 2000)
client.set_timeout(20.0)

In [2]:
# Global simulated V2X message broker queue for synchronization
v2x_broker_channel = queue.Queue(maxsize=20)

def roadside_unit_camera_loop(rsu_sensor, target_actors, stop_event):
    """
    Simulates a smart infrastructure RSU camera tracking multiple hidden VRUs.
    Broadcasts state estimates containing global coordinates and category tracking.
    """
    frame_queue = queue.Queue(maxsize=1)
    rsu_sensor.listen(lambda img: frame_queue.put(img) if not frame_queue.full() else None)
    
    while not stop_event.is_set():
        try:
            image = frame_queue.get(timeout=0.2)
            v2x_payload = []
            
            # Extract and package telemetry for all active hidden passengers/VRUs
            for actor in target_actors:
                if actor is not None and actor.is_alive:
                    tf = actor.get_transform()
                    vel = actor.get_velocity()
                    
                    v2x_payload.append({
                        "id": actor.id,
                        "type": "Cyclist" if "bicycle" in actor.type_id else "Pedestrian",
                        "pos_x": tf.location.x,
                        "pos_y": tf.location.y,
                        "pos_z": tf.location.z,
                        "speed": 3.6 * math.sqrt(vel.x**2 + vel.y**2 + vel.z**2)
                    })
            
            # Broadcast multi-object state tracking frame
            if not v2x_broker_channel.full():
                v2x_broker_channel.put({
                    "timestamp": image.timestamp,
                    "objects": v2x_payload
                })
                
            # Render infrastructure visualization feed
            arr = np.frombuffer(image.raw_data, dtype=np.uint8)
            arr = np.reshape(arr, (image.height, image.width, 4))
            bgr_frame = arr[:, :, :3].copy()
            cv2.putText(bgr_frame, f"RSU INFRASTRUCTURE TRACKING MESH: {len(v2x_payload)} TARGETS", 
                        (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
            cv2.imshow("RSU Infrastructure Perception Feed", bgr_frame)
            cv2.waitKey(1)
            
        except queue.Empty:
            continue
            
    rsu_sensor.stop()
    cv2.destroyWindow("RSU Infrastructure Perception Feed")

In [3]:
# Shared communication flag for live camera HUD overlay updates
v2x_hud_alert_triggered = False

def local_onboard_camera_loop(cam_sensor, stop_event, mode_string="UNASSISTED"):
    """Simulates an onboard front-facing safety camera with real-time HUD warning banners."""
    global v2x_hud_alert_triggered
    frame_queue = queue.Queue(maxsize=1)
    cam_sensor.listen(lambda img: frame_queue.put(img) if not frame_queue.full() else None)
    
    while not stop_event.is_set():
        try:
            image = frame_queue.get(timeout=0.2)
            arr = np.frombuffer(image.raw_data, dtype=np.uint8)
            arr = np.reshape(arr, (image.height, image.width, 4))
            bgr_frame = arr[:, :, :3].copy()
            
            # Draw standard system telemetry background
            cv2.rectangle(bgr_frame, (0, 0), (640, 60), (15, 15, 15), -1)
            
            if mode_string == "V2X-COOPERATIVE" and v2x_hud_alert_triggered:
                # Flashing High-Visibility HUD Threat Warning
                cv2.rectangle(bgr_frame, (5, 5), (635, 55), (0, 0, 255), 3)
                cv2.putText(bgr_frame, "⚠️ COLLISION RISK: HIDDEN VRU DETECTED BY RSU", 
                            (25, 37), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
            else:
                color = (0, 0, 255) if mode_string == "UNASSISTED" else (0, 255, 0)
                cv2.putText(bgr_frame, f"EGO LOCAL VIEW MODE: {mode_string}", 
                            (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
                            
            cv2.imshow("Ego Local View Window", bgr_frame)
            cv2.waitKey(1)
        except queue.Empty:
            continue
            
    cam_sensor.stop()
    cv2.destroyWindow("Ego Local View Window")

In [4]:
def run_assisted_scenario_v2(map_name, weather_preset, scenario_id):
    """
    Executes a test trial utilizing Cooperative V2X messaging sent from
    the roadside infrastructure camera to track a multi-passenger hidden crosswalk group.
    """
    global v2x_hud_alert_triggered
    v2x_hud_alert_triggered = False
    
    print(f"\n" + "="*80)
    print(f"STARTING ENHANCED V2X ASSISTED TRIAL: Map={map_name} | Scenario={scenario_id}")
    print("="*80)
    
    while not v2x_broker_channel.empty():
        v2x_broker_channel.get()
        
    world = client.load_world(map_name)
    world.set_weather(weather_preset)
    blueprint_library = world.get_blueprint_library()
    spectator = world.get_spectator()
    
    settings = world.get_settings()
    original_settings = world.get_settings()
    settings.synchronous_mode = True
    settings.fixed_delta_seconds = 0.04
    world.apply_settings(settings)
    
    trial_actors = []
    vru_group = []
    stop_signal = threading.Event()
    
    min_distance_to_target = float("inf")
    collision_detected = False
    speed_at_brake_trigger = 0.0
    brake_timestamp = None
    
    try:
        carla_map = world.get_map()
        spawn_points = carla_map.get_spawn_points()
        
        # Spatial setup configuration
        if "Town03" in map_name:
            ego_spawn_tf = carla.Transform(carla.Location(x=120.0, y=132.0, z=2.0), carla.Rotation(yaw=0.0))
            rsu_spawn_tf = carla.Transform(carla.Location(x=145.0, y=145.0, z=8.0), carla.Rotation(pitch=-35.0, yaw=-135.0, roll=0.0))
            
            # Establish multiple target spawn points to create a crowded blind crosswalk cluster
            vru_spawns = [
                carla.Transform(carla.Location(x=152.0, y=142.0, z=1.5), carla.Rotation(yaw=-90.0)), # Pedestrian 1
                carla.Transform(carla.Location(x=153.5, y=143.5, z=1.5), carla.Rotation(yaw=-90.0)), # Pedestrian 2
                carla.Transform(carla.Location(x=150.5, y=141.0, z=1.5), carla.Rotation(yaw=-90.0))  # Cyclist
            ]
            vru_direction = carla.Vector3D(0, -1, 0)
        else:
            ego_spawn_tf = spawn_points[0]
            rsu_spawn_tf = carla.Transform(ego_spawn_tf.location + carla.Location(x=20, y=10, z=8.0), carla.Rotation(pitch=-35.0, yaw=-90.0))
            vru_spawns = [carla.Transform(world.get_random_location_from_navigation() or spawn_points[5].location, carla.Rotation(yaw=0.0))]
            vru_direction = carla.Vector3D(-1, 0, 0)

        # Spawn Ego vehicle
        ego_vehicle = world.spawn_actor(blueprint_library.filter("vehicle.tesla.model3")[0], ego_spawn_tf)
        trial_actors.append(ego_vehicle)
        
        # Spawn Multiple VRU Targets (Pedestrians + Cyclists)
        ped_bp = blueprint_library.filter("walker.pedestrian.0001")[0]
        bike_bp = blueprint_library.filter("vehicle.bh.bicycle")[0]
        
        for i, spawn_tf in enumerate(vru_spawns):
            bp = bike_bp if i == 2 and "Town03" in map_name else ped_bp
            vru = world.spawn_actor(bp, spawn_tf)
            trial_actors.append(vru)
            vru_group.append(vru)
        
        # Setup Sensors
        ego_camera = world.spawn_actor(blueprint_library.find("sensor.camera.rgb"), carla.Transform(carla.Location(x=2.0, z=1.3)), attach_to=ego_vehicle)
        trial_actors.append(ego_camera)
        
        rsu_sensor = world.spawn_actor(blueprint_library.find("sensor.camera.rgb"), rsu_spawn_tf)
        trial_actors.append(rsu_sensor)
        
        # Threads
        cam_thread = threading.Thread(target=local_onboard_camera_loop, args=(ego_camera, stop_signal, "V2X-COOPERATIVE"), daemon=True)
        rsu_thread = threading.Thread(target=roadside_unit_camera_loop, args=(rsu_sensor, vru_group, stop_signal), daemon=True)
        cam_thread.start()
        rsu_thread.start()
        
        world.tick()
        time.sleep(0.5)
        
        # Initialize walking velocities across the cluster
        for vru in vru_group:
            if "bicycle" in vru.type_id:
                vru.apply_control(carla.VehicleControl(throttle=0.2)) # Handle bike controls as vehicles
            else:
                vru.apply_control(carla.WalkerControl(direction=vru_direction, speed=1.4 + random.uniform(-0.2, 0.2)))
        
        target_throttle, target_brake = 0.50, 0.0
        start_time = time.time()
        
        while time.time() - start_time < 12.0:
            world.tick()
            ego_tf = ego_vehicle.get_transform()
            move_spectator_to(ego_tf, spectator, distance=15.0, z=6.0, pitch=-22.0)
            
            # Proximity scanning across the target cluster
            current_closest = float("inf")
            for vru in vru_group:
                if vru.is_alive:
                    dist = ego_tf.location.distance(vru.get_transform().location)
                    if dist < current_closest: current_closest = dist
            
            if current_closest < min_distance_to_target: min_distance_to_target = current_closest
            if current_closest < 2.10: collision_detected = True
            
            vel = ego_vehicle.get_velocity()
            speed_kmh = 3.6 * math.sqrt(vel.x**2 + vel.y**2 + vel.z**2)
            
            # --- COMPREHENSIVE FUSION LOGIC ---
            v2x_alert_active = False
            if not v2x_broker_channel.empty():
                latest_packet = v2x_broker_channel.get()
                
                for obj in latest_packet["objects"]:
                    obj_loc = carla.Location(obj["pos_x"], obj["pos_y"], obj["pos_z"])
                    dist_to_ego = ego_tf.location.distance(obj_loc)
                    
                    # Intercept obstacle trajectories within a 30m hazard radius
                    if dist_to_ego < 30.0:
                        v2x_alert_active = True
                        # Real-time console status logger
                        print(f"[📡 V2X LOG] Ingress Threat ID {obj['id']} ({obj['type']}) | Range: {dist_to_ego:.1f}m | Target Speed: {obj['speed']:.1f} km/h")
            
            v2x_hud_alert_triggered = v2x_alert_active
            
            if v2x_alert_active:
                if target_brake == 0.0:
                    brake_timestamp = time.time() - start_time
                    speed_at_brake_trigger = speed_kmh
                    print(f"[🚨 POLICY] Early deceleration command initialized at t = {brake_timestamp:.2f}s")
                target_throttle, target_brake = 0.0, 0.70
                world.debug.draw_string(ego_tf.location + carla.Location(z=2.8), "📡 V2X NETWORK OVERLAY BRAKE", life_time=0.04, color=carla.Color(0,255,0))
            else:
                world.debug.draw_string(ego_tf.location + carla.Location(z=2.5), "CRUISE CONTROL ACTIVE", life_time=0.04, color=carla.Color(0,255,0))
                
            ego_vehicle.apply_control(carla.VehicleControl(throttle=float(target_throttle), brake=float(target_brake)))
            if target_brake > 0.0 and speed_kmh < 0.1:
                print("[🚨 POLICY] Safe early stop achieved before intersection boundary window.")
                break
            time.sleep(0.02)
            
    finally:
        stop_signal.set()
        cam_thread.join(timeout=1.5)
        rsu_thread.join(timeout=1.5)
        world.apply_settings(original_settings)
        safe_destroy(trial_actors)
        
    return {"collision": collision_detected, "min_dist": min_distance_to_target, "brake_time": brake_timestamp, "trigger_speed": speed_at_brake_trigger}

In [13]:
def run_assisted_scenario_v2(map_name, weather_preset, scenario_id):
    """
    Executes a test trial utilizing Cooperative V2X messaging sent from
    the roadside infrastructure camera to track a multi-passenger hidden crosswalk group.
    """
    global v2x_hud_alert_triggered
    v2x_hud_alert_triggered = False
    
    print(f"\n" + "="*80)
    print(f"STARTING ENHANCED V2X ASSISTED TRIAL: Map={map_name} | Scenario={scenario_id}")
    print("="*80)
    
    while not v2x_broker_channel.empty():
        v2x_broker_channel.get()
        
    world = client.load_world(map_name)
    world.set_weather(weather_preset)
    blueprint_library = world.get_blueprint_library()
    spectator = world.get_spectator()
    
    settings = world.get_settings()
    original_settings = world.get_settings()
    settings.synchronous_mode = True
    settings.fixed_delta_seconds = 0.04
    world.apply_settings(settings)
    
    trial_actors = []
    vru_group = []
    cam_thread = None
    rsu_thread = None
    stop_signal = threading.Event()
    
    min_distance_to_target = float("inf")
    collision_detected = False
    speed_at_brake_trigger = 0.0
    brake_timestamp = None
    
    try:
        carla_map = world.get_map()
        spawn_points = carla_map.get_spawn_points()
        
        if "Town03" in map_name:
            ego_spawn_tf = carla.Transform(carla.Location(x=120.0, y=132.0, z=2.0), carla.Rotation(yaw=0.0))
            rsu_spawn_tf = carla.Transform(carla.Location(x=145.0, y=145.0, z=8.0), carla.Rotation(pitch=-35.0, yaw=-135.0, roll=0.0))
            
            # --- FIX: Safe linear sequence spacing along an open stretch of the crosswalk line ---
            vru_direction = carla.Vector3D(0, -1, 0)
            vru_spawns = []
            base_x, base_y, base_z = 146.0, 140.0, 2.0
            
            for i in range(3):
                # Spread out along the X-axis with generous 2.5-meter gaps to prevent bounding box overlaps
                loc = carla.Location(x=base_x + (i * 2.5), y=base_y, z=base_z)
                vru_spawns.append(carla.Transform(loc, carla.Rotation(yaw=-90.0)))
        else:
            ego_spawn_tf = spawn_points[0]
            rsu_spawn_tf = carla.Transform(ego_spawn_tf.location + carla.Location(x=20, y=10, z=8.0), carla.Rotation(pitch=-35.0, yaw=-90.0))
            vru_spawns = [carla.Transform(world.get_random_location_from_navigation() or spawn_points[5].location, carla.Rotation(yaw=0.0))]
            vru_direction = carla.Vector3D(-1, 0, 0)

        # Spawn Ego vehicle
        ego_vehicle = world.spawn_actor(blueprint_library.filter("vehicle.tesla.model3")[0], ego_spawn_tf)
        trial_actors.append(ego_vehicle)
        
        # Blueprint Filtering
        ped_blueprints = blueprint_library.filter("walker.pedestrian.*")
        bike_blueprints = blueprint_library.filter("vehicle.bh.bicycle") or blueprint_library.filter("vehicle.diamondback.century") or blueprint_library.filter("vehicle.*")
        
        ped_bp = ped_blueprints[0]
        bike_bp = bike_blueprints[0] if bike_blueprints else ped_bp
        
        # --- FIX: Added try/except fallback within the spawn loop to bypass structural map conflicts dynamically ---
        for i, spawn_tf in enumerate(vru_spawns):
            bp = bike_bp if (i == 2 and "Town03" in map_name) else ped_bp
            try:
                vru = world.spawn_actor(bp, spawn_tf)
                trial_actors.append(vru)
                vru_group.append(vru)
            except RuntimeError as e:
                print(f"[⚠️ SPAWN WARNING] Direct placement failed at index {i}. Attempting navigation mesh fallback...")
                # Fallback to navigation mesh coordinate adjustments
                nav_loc = world.get_random_location_from_navigation()
                if nav_loc is not None:
                    spawn_tf.location = nav_loc
                    spawn_tf.location.z += 1.0
                    try:
                        vru = world.spawn_actor(bp, spawn_tf)
                        trial_actors.append(vru)
                        vru_group.append(vru)
                    except RuntimeError:
                        print(f"[❌ SPAWN ERROR] Fallback placement failed for index {i}. Skipping target.")
        
        if not vru_group:
            raise RuntimeError("Fatal Configuration Error: No target VRU actors successfully spawned inside the map environment.")

        # Setup Sensors
        ego_camera = world.spawn_actor(blueprint_library.find("sensor.camera.rgb"), carla.Transform(carla.Location(x=2.0, z=1.3)), attach_to=ego_vehicle)
        trial_actors.append(ego_camera)
        
        rsu_sensor = world.spawn_actor(blueprint_library.find("sensor.camera.rgb"), rsu_spawn_tf)
        trial_actors.append(rsu_sensor)
        
        # Launch Threads
        cam_thread = threading.Thread(target=local_onboard_camera_loop, args=(ego_camera, stop_signal, "V2X-COOPERATIVE"), daemon=True)
        rsu_thread = threading.Thread(target=roadside_unit_camera_loop, args=(rsu_sensor, vru_group, stop_signal), daemon=True)
        cam_thread.start()
        rsu_thread.start()
        
        world.tick()
        time.sleep(0.5)
        
        # Initialize movements
        for vru in vru_group:
            if "walker" not in vru.type_id:
                vru.apply_control(carla.VehicleControl(throttle=0.25)) 
            else:
                vru.apply_control(carla.WalkerControl(direction=vru_direction, speed=1.4 + random.uniform(-0.2, 0.2)))
        
        target_throttle, target_brake = 0.50, 0.0
        start_time = time.time()
        
        while time.time() - start_time < 12.0:
            world.tick()
            ego_tf = ego_vehicle.get_transform()
            move_spectator_to(ego_tf, spectator, distance=15.0, z=6.0, pitch=-22.0)
            
            current_closest = float("inf")
            for vru in vru_group:
                if vru.is_alive:
                    dist = ego_tf.location.distance(vru.get_transform().location)
                    if dist < current_closest: current_closest = dist
            
            if current_closest < min_distance_to_target: min_distance_to_target = current_closest
            if current_closest < 2.10: collision_detected = True
            
            vel = ego_vehicle.get_velocity()
            speed_kmh = 3.6 * math.sqrt(vel.x**2 + vel.y**2 + vel.z**2)
            
            # --- COMPREHENSIVE FUSION LOGIC ---
            v2x_alert_active = False
            if not v2x_broker_channel.empty():
                latest_packet = v2x_broker_channel.get()
                
                for obj in latest_packet["objects"]:
                    obj_loc = carla.Location(obj["pos_x"], obj["pos_y"], obj["pos_z"])
                    dist_to_ego = ego_tf.location.distance(obj_loc)
                    
                    if dist_to_ego < 30.0:
                        v2x_alert_active = True
                        print(f"[📡 V2X LOG] Ingress Threat ID {obj['id']} ({obj['type']}) | Range: {dist_to_ego:.1f}m | Target Speed: {obj['speed']:.1f} km/h")
            
            v2x_hud_alert_triggered = v2x_alert_active
            
            if v2x_alert_active:
                if target_brake == 0.0:
                    brake_timestamp = time.time() - start_time
                    speed_at_brake_trigger = speed_kmh
                    print(f"[🚨 POLICY] Early deceleration command initialized at t = {brake_timestamp:.2f}s")
                target_throttle, target_brake = 0.0, 0.70
                world.debug.draw_string(ego_tf.location + carla.Location(z=2.8), "📡 V2X NETWORK OVERLAY BRAKE", life_time=0.04, color=carla.Color(0,255,0))
            else:
                world.debug.draw_string(ego_tf.location + carla.Location(z=2.5), "CRUISE CONTROL ACTIVE", life_time=0.04, color=carla.Color(0,255,0))
                
            ego_vehicle.apply_control(carla.VehicleControl(throttle=float(target_throttle), brake=float(target_brake)))
            if target_brake > 0.0 and speed_kmh < 0.1:
                print("[🚨 POLICY] Safe early stop achieved before intersection boundary window.")
                break
            time.sleep(0.02)
            
    finally:
        stop_signal.set()
        if cam_thread is not None:
            cam_thread.join(timeout=1.5)
        if rsu_thread is not None:
            rsu_thread.join(timeout=1.5)
        world.apply_settings(original_settings)
        safe_destroy(trial_actors)
        
    return {"collision": collision_detected, "min_dist": min_distance_to_target, "brake_time": brake_timestamp, "trigger_speed": speed_at_brake_trigger}

In [14]:
# Execute Baseline Configurations
unassisted_r1 = run_assisted_scenario_v2("Town03", carla.WeatherParameters.ClearNoon, 1)
unassisted_r2 = run_assisted_scenario_v2("Town05", carla.WeatherParameters.HardRainSunset, 2)

# Execute V2X Enhanced Configurations
assisted_r1 = run_assisted_scenario("Town03", carla.WeatherParameters.ClearNoon, 1)
assisted_r2 = run_assisted_scenario("Town05", carla.WeatherParameters.HardRainSunset, 2)

# Generate Quantitative Benchmarking Table Output
print("\n" + "="*90)
print("                    PROJECT 5 DELIVERABLE: COOPERATIVE PERCEPTION REPORT                   ")
print("="*90)
print(f"{'Performance Metric':<32} | {'S1 Town03 (Base)':<16} | {'S1 Town03 (V2X)':<15} | {'S2 Town05 (Base)':<16} | {'S2 Town05 (V2X)':<15}")
print("-"*90)

o_u1 = "💥 COLLISION" if unassisted_r1["collision"] else "✅ AVOIDED"
o_a1 = "💥 COLLISION" if assisted_r1["collision"] else "✅ AVOIDED"
o_u2 = "💥 COLLISION" if unassisted_r2["collision"] else "✅ AVOIDED"
o_a2 = "💥 COLLISION" if assisted_r2["collision"] else "✅ AVOIDED"
print(f"{'Safety Validation Outcome':<32} | {o_u1:<16} | {o_a1:<15} | {o_u2:<16} | {o_a2:<15}")

print(f"{'Minimum Spatial Proximity (m)':<32} | {unassisted_r1['min_dist']:<16.2f} | {assisted_r1['min_dist']:<15.2f} | {unassisted_r2['min_dist']:<16.2f} | {assisted_r2['min_dist']:<15.2f}")

t_u1 = f"{unassisted_r1['brake_time']:.2f}s" if unassisted_r1['brake_time'] else "N/A"
t_a1 = f"{assisted_r1['brake_time']:.2f}s" if assisted_r1['brake_time'] else "N/A"
t_u2 = f"{unassisted_r2['brake_time']:.2f}s" if unassisted_r2['brake_time'] else "N/A"
t_a2 = f"{assisted_r2['brake_time']:.2f}s" if assisted_r2['brake_time'] else "N/A"
print(f"{'Perception Brake Trigger Time':<32} | {t_u1:<16} | {t_a1:<15} | {t_u2:<16} | {t_a2:<15}")

# Compute Warning Lead Time Gains
gain_s1 = unassisted_r1['brake_time'] - assisted_r1['brake_time'] if (unassisted_r1['brake_time'] and assisted_r1['brake_time']) else 0.0
gain_s2 = unassisted_r2['brake_time'] - assisted_r2['brake_time'] if (unassisted_r2['brake_time'] and assisted_r2['brake_time']) else 0.0
print(f"{'V2X Warning Lead Time Margin':<32} | {'-':<16} | {f'+{gain_s1:.2f}s':<15} | {'-':<16} | {f'+{gain_s2:.2f}s':<15}")
print("="*90)


STARTING ENHANCED V2X ASSISTED TRIAL: Map=Town03 | Scenario=1
[⚠️ SPAWN WARNING] Direct placement failed at index 1. Attempting navigation mesh fallback...
[❌ SPAWN ERROR] Fallback placement failed for index 1. Skipping target.
[⚠️ SPAWN WARNING] Direct placement failed at index 2. Attempting navigation mesh fallback...
[📡 V2X LOG] Ingress Threat ID 989 (Pedestrian) | Range: 27.2m | Target Speed: 4.2 km/h
[🚨 POLICY] Early deceleration command initialized at t = 0.28s
[📡 V2X LOG] Ingress Threat ID 989 (Pedestrian) | Range: 27.2m | Target Speed: 7.1 km/h
[📡 V2X LOG] Ingress Threat ID 989 (Pedestrian) | Range: 27.2m | Target Speed: 8.5 km/h
[📡 V2X LOG] Ingress Threat ID 989 (Pedestrian) | Range: 27.2m | Target Speed: 9.9 km/h
[📡 V2X LOG] Ingress Threat ID 989 (Pedestrian) | Range: 27.2m | Target Speed: 9.9 km/h
[📡 V2X LOG] Ingress Threat ID 989 (Pedestrian) | Range: 27.2m | Target Speed: 9.9 km/h
[📡 V2X LOG] Ingress Threat ID 989 (Pedestrian) | Range: 27.2m | Target Speed: 9.9 km/h
[📡 V2

Exception in thread Thread-10:
Traceback (most recent call last):
  File "c:\anaconda3\envs\carla-env\lib\threading.py", line 917, in _bootstrap_inner
    self.run()
  File "c:\anaconda3\envs\carla-env\lib\threading.py", line 865, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\danial.khayatian\AppData\Local\Temp\ipykernel_17652\1225203325.py", line 18, in roadside_unit_camera_loop
    for actor in target_actors:
TypeError: 'Walker' object is not iterable




LAUNCHING V2X-ASSISTED SCENARIO 2...


Exception in thread Thread-12:
Traceback (most recent call last):
  File "c:\anaconda3\envs\carla-env\lib\threading.py", line 917, in _bootstrap_inner
    self.run()
  File "c:\anaconda3\envs\carla-env\lib\threading.py", line 865, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\danial.khayatian\AppData\Local\Temp\ipykernel_17652\1225203325.py", line 18, in roadside_unit_camera_loop
    for actor in target_actors:
TypeError: 'Walker' object is not iterable




                    PROJECT 5 DELIVERABLE: COOPERATIVE PERCEPTION REPORT                   
Performance Metric               | S1 Town03 (Base) | S1 Town03 (V2X) | S2 Town05 (Base) | S2 Town05 (V2X)
------------------------------------------------------------------------------------------
Safety Validation Outcome        | ✅ AVOIDED        | ✅ AVOIDED       | ✅ AVOIDED        | ✅ AVOIDED      
Minimum Spatial Proximity (m)    | 27.01            | 6.79            | 276.03           | 69.80          
Perception Brake Trigger Time    | 0.28s            | N/A             | N/A              | N/A            
V2X Warning Lead Time Margin     | -                | +0.00s          | -                | +0.00s         
